# 0.16 — Unseen edge emergence (genAI benchmark)

Follow-up to [`0.15`](0.15-genai-novelty-emergence.ipynb) Part A: detect emerging themes via **baseline-unseen vocabulary + pair bursts**, building **micro-graphs** from top emerging edges only — not full-corpus or residual PPMI Louvain.

**Question:** Can unsupervised ranking of **unseen-in-baseline edges with weekly burst + acceleration** surface the genAI cluster in **Dec 2022–Jan 2023**?

**Avoids (lessons from 0.13/0.14):** full weekly PPMI Louvain · headline embedding pre-filter · raw `-log P` alone.

**GenAI lexicon:** validation/annotation only (same as 0.13–0.15).

**Prerequisites:** `genai_graph_terms.parquet` ([`0.13`](0.13-genai-graph-emergence.ipynb)); `genai_full_meta.parquet` ([`0.10`](0.10-genai-hierarchy-drilldown.ipynb)).

**Milestones (annotation only):** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

In [1]:
import re
from collections import Counter, defaultdict
from itertools import combinations
from math import log, sqrt
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from networkx.algorithms.community import louvain_communities
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tqdm.auto import tqdm

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- time windows ---
BASELINE_END = pd.Timestamp("2022-09-30")
DISCOVERY_START = pd.Timestamp("2022-12-01")
DISCOVERY_END = pd.Timestamp("2023-01-31")
DATE_END = pd.Timestamp("2023-01-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")
RANDOM_SEED = 42

META_CACHE = OUTPUT_DIR / "genai_full_meta.parquet"
TERMS_CACHE = OUTPUT_DIR / "genai_graph_terms.parquet"
RANKINGS_013 = OUTPUT_DIR / "genai_graph_emergence_rankings.parquet"
VALIDATION_013 = OUTPUT_DIR / "genai_graph_validation.parquet"
RANKINGS_014 = OUTPUT_DIR / "genai_residual_emergence_rankings.parquet"
VALIDATION_014 = OUTPUT_DIR / "genai_residual_validation.parquet"

WEEKLY_TERMS_PATH = OUTPUT_DIR / "genai_unseen_weekly_terms.parquet"
WEEKLY_EDGES_PATH = OUTPUT_DIR / "genai_unseen_weekly_edges.parquet"
COMMUNITIES_PATH = OUTPUT_DIR / "genai_unseen_communities.parquet"
RANKINGS_PATH = OUTPUT_DIR / "genai_unseen_rankings.parquet"
VALIDATION_PATH = OUTPUT_DIR / "genai_unseen_validation.parquet"
COMPARE_PATH = OUTPUT_DIR / "genai_unseen_compare_methods.parquet"
TIMELINE_PATH = OUTPUT_DIR / "genai_unseen_timeline.html"
HEATMAP_PATH = OUTPUT_DIR / "genai_unseen_edge_heatmap.html"
MICROGRAPH_PATH = OUTPUT_DIR / "genai_unseen_micrograph.html"
DASHBOARD_PATH = OUTPUT_DIR / "genai_unseen_dashboard.html"

FREQ = "W-MON"
SMOOTH_ALPHA = 1.0
ALPHA = 1.0
MIN_UNSEEN_COOC = 2
TOP_EDGES_PER_WEEK = 100
MIN_EDGE_BURST = 0.0
MIN_TERM_BURST = 0.0
WOW_MIN = 1.5
WOW_CAP = 10.0
MIN_COMMUNITY_SIZE = 3
MIN_INTERNAL_UNSEEN_EDGES = 2
JACCARD_LINK = 0.25
LOUVAIN_RESOLUTION = 1.0
LOUVAIN_SEED = 42
MIN_HEADLINES = 5
TOP_TERMS = 12
TOP_HEADLINES = 10
TOP_RANK_PRINT = 15
TOP_N = 15
HEATMAP_TOP_EDGES = 30

FINANCE_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock", "stocks",
    "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan", "bn",
    "march", "april", "june", "july", "august", "september", "october", "november", "december",
    "rated", "buy", "sell", "hold", "neutral", "perform", "outperform", "underperform",
    "overweight", "underweight", "equal-weight", "cut", "raise", "raised", "lowers", "upgrade",
    "downgrade", "maintains", "reiterates", "est", "eps", "adj", "sees", "expects", "forecast",
    "names", "appoints", "hires", "officer", "director", "chairman", "executive", "promotes",
    "tender", "offering", "offer", "notes", "bond", "bonds", "debt", "bills", "yield",
}
TERM_STOP = FINANCE_STOP

GENAI_T1 = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle",
]
GENAI_T2 = [
    r"\bopenai\b", r"\banthropic\b", r"chatgpt-like", r"chatgpt-style",
    r"prompt engineering", r"ai chatbot", r"ai chat bot",
]
GENAI_STANDARD = re.compile("|".join(f"(?:{p})" for p in GENAI_T1 + GENAI_T2), re.I)
GENAI_TERM_HINT = re.compile(
    r"chatgpt|openai|chatbot|generative|anthropic|\bllm\b|\bllms\b|copilot|dall-e|dalle|midjourney|stable.?diffusion|foundation.?model|gpt-\d|gpt-3|gpt-4|\bgpt\b|bard|gemini|bing",
    re.I,
)


def normalize_terms(value) -> list:
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple)):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return []


def filter_terms_list(terms: list[str]) -> list[str]:
    out = []
    for t in terms:
        if any(tok in TERM_STOP for tok in t.split()):
            continue
        if len(t) < 3:
            continue
        out.append(t)
    return out


def week_ts(week: str) -> pd.Timestamp:
    return pd.Period(week, freq=FREQ).start_time


def ek(a: str, b: str) -> tuple[str, str]:
    return (a, b) if a < b else (b, a)


def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if (a or b) else 0.0


def add_date_marker(fig, ts, text: str = "") -> None:
    x = pd.Timestamp(ts)
    fig.add_shape(
        type="line", x0=x, x1=x, y0=0, y1=1, yref="paper",
        line=dict(dash="dash", color="gray", width=1),
    )
    if text:
        fig.add_annotation(x=x, y=1.04, yref="paper", text=text, showarrow=False, font=dict(size=10))


def zscore_series(s: pd.Series) -> pd.Series:
    if s.std(ddof=0) == 0 or len(s) < 2:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / s.std(ddof=0)


print(f"0.16 unseen edge emergence | baseline through {BASELINE_END.date()}")
print(f"Discovery {DISCOVERY_START.date()}→{DISCOVERY_END.date()}")

0.16 unseen edge emergence | baseline through 2022-09-30
Discovery 2022-12-01→2023-01-31


## §1 — Load data & baseline statistics

In [2]:
meta = pd.read_parquet(META_CACHE).reset_index(drop=True)
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
meta = meta.loc[meta.date <= DATE_END].copy()
meta["row_id"] = np.arange(len(meta))

terms = pd.read_parquet(TERMS_CACHE)
terms["date"] = pd.to_datetime(terms["date"]).dt.normalize()
terms = terms.drop_duplicates(["Headline", "date"], keep="first")

news = meta.merge(terms[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news["terms"] = news["terms"].apply(normalize_terms)
news["terms_f"] = news["terms"].map(filter_terms_list)
news["week"] = news["date"].dt.to_period(FREQ).astype(str)
news["hl_lc"] = news["Headline"].str.lower()
news["is_genai"] = news["hl_lc"].str.contains(GENAI_STANDARD, na=False)

baseline_df = news.loc[news.date <= BASELINE_END]
baseline_n_docs = len(baseline_df)
baseline_term_df = Counter()
for tlist in baseline_df["terms_f"]:
    baseline_term_df.update(set(tlist))


def count_pairs(rows: list[list[str]]) -> Counter:
    pc = Counter()
    for terms in rows:
        u = sorted(set(terms))
        pc.update(combinations(u, 2))
    return pc


baseline_pair_df = count_pairs(baseline_df["terms_f"].tolist())
n_pairs_vocab = len(baseline_pair_df)


def is_unseen_term(term: str) -> bool:
    return term not in baseline_term_df


def is_unseen_pair(a: str, b: str) -> bool:
    return ek(a, b) not in baseline_pair_df


def word_novelty(term: str) -> float:
    c = baseline_term_df.get(term, 0)
    V = len(baseline_term_df)
    p = (c + SMOOTH_ALPHA) / (baseline_n_docs + V * SMOOTH_ALPHA)
    return -log(p)


def edge_novelty(a: str, b: str) -> float:
    key = ek(a, b)
    c = baseline_pair_df.get(key, 0)
    p = (c + SMOOTH_ALPHA) / (baseline_n_docs + n_pairs_vocab * SMOOTH_ALPHA)
    return -log(p)


# weekly term / edge counts for burst denominators
twc: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
ewc: dict[tuple[str, str], dict[str, int]] = defaultdict(lambda: defaultdict(int))
week_headline_counts: dict[str, int] = {}

for week, grp in tqdm(news.groupby("week"), desc="weekly counts"):
    week_headline_counts[week] = len(grp)
    for tlist in grp["terms_f"]:
        for t in set(tlist):
            twc[t][week] += 1
        for a, b in combinations(sorted(set(tlist)), 2):
            ewc[ek(a, b)][week] += 1

all_weeks = sorted(news["week"].unique(), key=week_ts)
baseline_weeks = [w for w in all_weeks if week_ts(w) <= BASELINE_END]
discovery_weeks = [w for w in all_weeks if DISCOVERY_START <= week_ts(w) <= DISCOVERY_END]


def bexp(counts: dict, weeks: list[str]) -> float:
    return float(np.mean([counts.get(w, 0) for w in weeks])) if weeks else 0.0


def term_burst(term: str, week: str) -> float:
    return log((twc[term].get(week, 0) + ALPHA) / (bexp(twc[term], baseline_weeks) + ALPHA))


def edge_burst(a: str, b: str, week: str) -> float:
    key = ek(a, b)
    return log((ewc[key].get(week, 0) + ALPHA) / (bexp(ewc[key], baseline_weeks) + ALPHA))


print(f"Headlines through {DATE_END.date()}: {len(news):,}")
print(f"Baseline docs: {baseline_n_docs:,} | vocab: {len(baseline_term_df):,} | pairs: {len(baseline_pair_df):,}")
print(f"Discovery weeks: {len(discovery_weeks)} ({discovery_weeks[0]} … {discovery_weeks[-1]})")

weekly counts:   0%|          | 0/110 [00:00<?, ?it/s]

Headlines through 2023-01-31: 2,106,318
Baseline docs: 1,806,435 | vocab: 983,360 | pairs: 16,506,789
Discovery weeks: 9 (2022-12-06/2022-12-12 … 2023-01-31/2023-02-06)


## §2 — Weekly unseen term census

In [3]:
term_rows = []
prior_term_df: dict[str, int] = {}

for wi, week in enumerate(discovery_weeks):
    grp = news.loc[news.week == week]
    tc = Counter()
    for tlist in grp["terms_f"]:
        tc.update(set(tlist))
    prev_week = discovery_weeks[wi - 1] if wi > 0 else None
    if prev_week:
        prev_grp = news.loc[news.week == prev_week]
        prior_term_df = Counter()
        for tlist in prev_grp["terms_f"]:
            prior_term_df.update(set(tlist))
    week_candidates = []
    for term, doc_freq in tc.items():
        if not is_unseen_term(term):
            continue
        if doc_freq < MIN_UNSEEN_COOC:
            continue
        tb = term_burst(term, week)
        if tb < MIN_TERM_BURST:
            continue
        prev_df = prior_term_df.get(term, 0) if prev_week else 0
        wow_ratio = doc_freq / max(prev_df, 1)
        term_score = tb * log(1 + doc_freq) * min(wow_ratio, WOW_CAP)
        week_candidates.append({
            "week": week,
            "term": term,
            "doc_freq": doc_freq,
            "term_burst": round(tb, 4),
            "wow_ratio": round(wow_ratio, 4),
            "word_novelty": round(word_novelty(term), 4),
            "term_score": round(term_score, 4),
        })
    week_candidates.sort(key=lambda r: (-r["term_score"], -r["word_novelty"]))
    for rank, row in enumerate(week_candidates, 1):
        row["rank_in_week"] = rank
        term_rows.append(row)
    print(f"\n=== Top {TOP_N} unseen terms — {week} ===")
    if week_candidates:
        print(pd.DataFrame(week_candidates[:TOP_N]).to_string(index=False))
    else:
        print("  (none)")

terms_df = pd.DataFrame(term_rows)
terms_df.to_parquet(WEEKLY_TERMS_PATH, index=False)
print(f"\nWrote {WEEKLY_TERMS_PATH.name}: {len(terms_df):,} rows")


=== Top 15 unseen terms — 2022-12-06/2022-12-12 ===
                 week               term  doc_freq  term_burst  wow_ratio  word_novelty  term_score  rank_in_week
2022-12-06/2022-12-12    block microsoft        14      2.7081       14.0       14.8415     73.3354             1
2022-12-06/2022-12-12  baptista research        10      2.3979       10.0       14.8415     57.4990             2
2022-12-06/2022-12-12           baptista        10      2.3979       10.0       14.8415     57.4990             3
2022-12-06/2022-12-12          crestchic         7      2.0794        7.0       14.8415     30.2685             4
2022-12-06/2022-12-12         orbusneich         7      2.0794        7.0       14.8415     30.2685             5
2022-12-06/2022-12-12 orbusneich medical         7      2.0794        7.0       14.8415     30.2685             6
2022-12-06/2022-12-12   prevent creditor         6      1.9459        6.0       14.8415     22.7194             7
2022-12-06/2022-12-12       mng air

## §3 — Weekly unseen edge census (primary signal)

In [4]:
edge_rows = []
prior_pair_counts: Counter = Counter()

for wi, week in enumerate(discovery_weeks):
    grp = news.loc[news.week == week]
    pc = count_pairs(grp["terms_f"].tolist())
    if wi > 0:
        prev_grp = news.loc[news.week == discovery_weeks[wi - 1]]
        prior_pair_counts = count_pairs(prev_grp["terms_f"].tolist())
    week_candidates = []
    for (a, b), cooc_count in pc.items():
        if not is_unseen_pair(a, b):
            continue
        if cooc_count < MIN_UNSEEN_COOC:
            continue
        eb = edge_burst(a, b, week)
        if eb < MIN_EDGE_BURST:
            continue
        prev_cooc = prior_pair_counts.get((a, b), 0)
        wow_ratio = cooc_count / max(prev_cooc, 1)
        if wow_ratio < WOW_MIN and cooc_count < 5:
            continue
        edge_score = eb * log(1 + cooc_count) * min(wow_ratio, WOW_CAP)
        week_candidates.append({
            "week": week,
            "term_a": a,
            "term_b": b,
            "cooc_count": cooc_count,
            "edge_burst": round(eb, 4),
            "wow_ratio": round(wow_ratio, 4),
            "edge_novelty": round(edge_novelty(a, b), 4),
            "edge_score": round(edge_score, 4),
        })
    week_candidates.sort(key=lambda r: (-r["edge_score"], -r["edge_novelty"]))
    for rank, row in enumerate(week_candidates, 1):
        row["rank_in_week"] = rank
        edge_rows.append(row)
    print(f"\n=== Top {TOP_N} unseen edges — {week} ({len(week_candidates)} total) ===")
    if week_candidates:
        print(pd.DataFrame(week_candidates[:TOP_N]).to_string(index=False))
    else:
        print("  (none)")

edges_df = pd.DataFrame(edge_rows)
edges_df.to_parquet(WEEKLY_EDGES_PATH, index=False)
print(f"\nWrote {WEEKLY_EDGES_PATH.name}: {len(edges_df):,} rows")


=== Top 15 unseen edges — 2022-12-06/2022-12-12 (22775 total) ===
                 week          term_a          term_b  cooc_count  edge_burst  wow_ratio  edge_novelty  edge_score  rank_in_week
2022-12-06/2022-12-12           amgen         horizon          37      3.6376       37.0       16.7231    132.3203             1
2022-12-06/2022-12-12             nrg          vivint          24      3.2189       24.0       16.7231    103.6116             2
2022-12-06/2022-12-12             chr       novozymes          22      3.1355       22.0       16.7231     98.3132             3
2022-12-06/2022-12-12          hansen       novozymes          22      3.1355       22.0       16.7231     98.3132             4
2022-12-06/2022-12-12           coupa           thoma          21      3.0910       21.0       16.7231     95.5454             5
2022-12-06/2022-12-12           bravo           coupa          20      3.0445       20.0       16.7231     92.6912             6
2022-12-06/2022-12-12         

## §4–6 — Micro-graph, Louvain, tracking & S* ranking

In [5]:
def contrast_top_terms(terms: set[str], week: str, member_indices: list[int]) -> list[str]:
    if not terms:
        return []
    in_ctr, out_ctr = Counter(), Counter()
    grp = news.loc[news.week == week]
    comm_set = set(member_indices)
    for idx, tlist in zip(grp.index, grp["terms_f"]):
        ts = set(tlist) & terms
        tgt = in_ctr if idx in comm_set else out_ctr
        for t in ts:
            tgt[t] += 1
    n_in, n_out = max(len(comm_set), 1), max(len(grp) - len(comm_set), 1)
    scored = []
    for t in terms:
        p_in = (in_ctr[t] + ALPHA) / (n_in + ALPHA)
        p_out = (out_ctr[t] + ALPHA) / (n_out + ALPHA)
        scored.append((log(p_in / p_out), t))
    scored.sort(reverse=True)
    return [t for _, t in scored[:TOP_TERMS]]


def community_coherence(g: nx.Graph, terms: set[str]) -> float:
    ws = [g[a][b].get("weight", 0) if g.has_edge(a, b) else 0 for a, b in combinations(sorted(terms), 2)]
    return float(np.mean(ws)) if ws else 0.0


seed_edges_by_week: dict[str, list[tuple]] = {}
micro_graphs: dict[str, nx.Graph] = {}
communities_by_week: dict[str, list[dict]] = {}

for week in discovery_weeks:
    wk_edges = edges_df.loc[edges_df.week == week].sort_values("edge_score", ascending=False).head(TOP_EDGES_PER_WEEK)
    if wk_edges.empty:
        communities_by_week[week] = []
        continue
    g = nx.Graph()
    seed_list = []
    for _, r in wk_edges.iterrows():
        a, b = r.term_a, r.term_b
        w = float(r.edge_score)
        g.add_edge(a, b, weight=w, edge_score=w, cooc_count=int(r.cooc_count))
        seed_list.append((a, b, w))
    for node in g.nodes:
        g.nodes[node]["doc_freq"] = twc[node].get(week, 0)
        g.nodes[node]["term_burst"] = term_burst(node, week)
        g.nodes[node]["unseen"] = is_unseen_term(node)
    if g.number_of_nodes() < 3 or g.number_of_edges() < 2:
        communities_by_week[week] = []
        continue
    seed_edges_by_week[week] = seed_list
    micro_graphs[week] = g
    raw_comms = []
    for idx, term_set in enumerate(louvain_communities(g, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=LOUVAIN_SEED)):
        terms = set(term_set)
        if len(terms) < MIN_COMMUNITY_SIZE:
            continue
        internal = [(a, b, g[a][b].get("edge_score", 0)) for a, b in combinations(sorted(terms), 2) if g.has_edge(a, b)]
        if len(internal) < MIN_INTERNAL_UNSEEN_EDGES:
            continue
        raw_comms.append({
            "community_id": idx,
            "terms": terms,
            "internal_edges": internal,
            "coherence": community_coherence(g, terms),
            "mean_edge_score": float(np.mean([e[2] for e in internal])),
            "max_edge_score": float(max(e[2] for e in internal)),
            "n_unseen_internal_edges": len(internal),
        })
    communities_by_week[week] = raw_comms

print(f"Micro-graph weeks: {len(micro_graphs)} | communities: {sum(len(v) for v in communities_by_week.values())}")

# --- track communities ---
next_track = 0
prev_comms: list[dict] = []
track_records: list[dict] = []
track_baseline_shares: dict[int, list[float]] = defaultdict(list)
min_share_prior = 1.0 / max(week_headline_counts.values())

for week in discovery_weeks:
    grp = news.loc[news.week == week]
    hl_idx = grp.index.tolist()
    hl_terms = grp["terms_f"].tolist()
    week_total = week_headline_counts[week]
    matched_prev: set[int] = set()
    week_comms: list[dict] = []

    for c in communities_by_week.get(week, []):
        best_j, best_track = 0.0, None
        for pc in prev_comms:
            if pc["track_id"] in matched_prev:
                continue
            j = jaccard(c["terms"], pc["terms"])
            if j >= JACCARD_LINK and j > best_j:
                best_j, best_track = j, pc["track_id"]
        if best_track is None:
            best_track = next_track
            next_track += 1
        else:
            matched_prev.add(best_track)

        members = [i for i, ts in zip(hl_idx, hl_terms) if c["terms"] & set(ts)]
        tb_vals = [term_burst(t, week) for t in c["terms"]]
        rec = dict(c)
        rec.update({
            "track_id": best_track,
            "period": week,
            "headline_count": len(members),
            "headline_indices": members,
            "mean_term_burst": float(np.mean(tb_vals)) if tb_vals else 0.0,
            "vocabulary_novelty": float(np.mean([1.0 if is_unseen_term(t) else 0.0 for t in c["terms"]])),
            "top_terms": contrast_top_terms(c["terms"], week, members),
        })
        share = rec["headline_count"] / week_total if week_total else 0.0
        rec["share"] = share
        track_baseline_shares[best_track].append(share)
        week_comms.append(rec)
        track_records.append(rec)
    prev_comms = week_comms

track_weeks: dict[int, list[str]] = defaultdict(list)
for c in track_records:
    track_weeks[c["track_id"]].append(c["period"])

for c in track_records:
    tid = c["track_id"]
    base_shares = track_baseline_shares.get(tid, [])
    p_bar = float(np.mean(base_shares[: max(len(base_shares) - 1, 1)])) if len(base_shares) > 1 else min_share_prior
    c["share_growth"] = log((c["share"] + ALPHA) / (p_bar + ALPHA))
    ws = sorted(track_weeks[tid], key=week_ts)
    streak = 1
    for i in range(1, len(ws)):
        if (week_ts(ws[i]) - week_ts(ws[i - 1])).days <= 8:
            streak += 1
        else:
            break
    c["persistence"] = streak
    c["first_seen"] = ws[0]

print(f"Track records: {len(track_records):,} | tracks: {next_track:,}")

# --- S* ranking ---
POS_FEATURES = [
    "mean_edge_score", "n_unseen_internal_edges", "mean_term_burst",
    "vocabulary_novelty", "coherence", "share_growth", "persistence",
]
rows = []
for c in track_records:
    rows.append({
        "period": c["period"], "community_id": c["community_id"], "track_id": c["track_id"],
        "top_terms": ", ".join(c["top_terms"]), "headline_count": c["headline_count"],
        "share": c["share"], "first_seen": c["first_seen"],
        **{f: c[f] for f in POS_FEATURES},
    })
rank_df = pd.DataFrame(rows)
rank_df["S"] = 0.0
rank_df["S_star"] = 0.0
reliability_denom = log(1 + MIN_HEADLINES)

for week, sub in rank_df.groupby("period"):
    z_pos = np.vstack([zscore_series(sub[f]) for f in POS_FEATURES])
    s = np.mean(z_pos, axis=0)
    rel = np.minimum(1.0, np.log1p(sub["headline_count"].values) / reliability_denom)
    rank_df.loc[sub.index, "S"] = s
    rank_df.loc[sub.index, "S_star"] = s * rel

rank_df["rank_in_week"] = rank_df.groupby("period")["S_star"].rank(ascending=False, method="first").astype(int)
rank_df = rank_df.sort_values(["period", "rank_in_week"]).reset_index(drop=True)
rank_df.to_parquet(RANKINGS_PATH, index=False)

comm_df = pd.DataFrame([{
    "period": c["period"], "community_id": c["community_id"], "track_id": c["track_id"],
    "top_terms": ", ".join(c["top_terms"]), "headline_count": c["headline_count"],
    "share": c["share"], "share_growth": c["share_growth"], "mean_edge_score": c["mean_edge_score"],
    "n_unseen_internal_edges": c["n_unseen_internal_edges"], "coherence": c["coherence"],
    "persistence": c["persistence"], "first_seen": c["first_seen"],
} for c in track_records])
comm_df.to_parquet(COMMUNITIES_PATH, index=False)

comm_lookup = {(c["period"], c["track_id"]): c for c in track_records}
rep_rows = []
for _, r in rank_df.loc[rank_df.rank_in_week <= TOP_RANK_PRINT].iterrows():
    c = comm_lookup.get((r["period"], r["track_id"]))
    if c is None:
        continue
    mg = micro_graphs.get(r["period"])
    term_w = {}
    if mg:
        for t in c["terms"]:
            term_w[t] = (mg.degree(t, weight="weight") if t in mg else 0) * np.exp(term_burst(t, r["period"]))
    scores = []
    for idx in c["headline_indices"][:5000]:
        score = sum(term_w.get(t, 0) for t in set(news.at[idx, "terms_f"]) & c["terms"])
        scores.append((score, news.at[idx, "Headline"]))
    scores.sort(key=lambda x: -x[0])
    for rep_rank, (score, hl) in enumerate(scores[:TOP_HEADLINES], 1):
        rep_rows.append({
            "period": r["period"], "track_id": r["track_id"], "rank_in_week": int(r["rank_in_week"]),
            "rep_rank": rep_rank, "score": score, "headline": hl[:240],
        })
rep_df = pd.DataFrame(rep_rows)

print(f"Wrote {RANKINGS_PATH.name}: {len(rank_df):,} rows")
print(f"Wrote {COMMUNITIES_PATH.name}: {len(comm_df):,} rows")
print("\nTop S* communities — discovery window:")
for week in discovery_weeks:
    sub = rank_df.loc[rank_df.period == week].head(5)
    if sub.empty:
        continue
    print(f"\n  {week}")
    for _, r in sub.iterrows():
        print(f"    #{int(r.rank_in_week)} track {int(r.track_id)} S*={r.S_star:.2f}  {r.top_terms[:70]}")

Micro-graph weeks: 9 | communities: 165
Track records: 165 | tracks: 162
Wrote genai_unseen_rankings.parquet: 165 rows
Wrote genai_unseen_communities.parquet: 165 rows

Top S* communities — discovery window:

  2022-12-06/2022-12-12
    #1 track 2 S*=0.68  novozymes, hansen, chr
    #2 track 8 S*=0.66  research, baptista research, baptista
    #3 track 1 S*=0.51  energy, home, nrg, vivint, smart, nrg energy, smart home
    #4 track 3 S*=0.39  coupa, thoma, bravo, thoma bravo
    #5 track 0 S*=0.33  amgen, horizon, sanofi

  2022-12-13/2022-12-19
    #1 track 20 S*=1.29  bank, ftx, fraud, danske, danske bank, guilty, forfeit, pleads guilty,
    #2 track 19 S*=1.12  probe, budget, pldt, overrun, budget overrun
    #3 track 18 S*=0.72  bankman-fried, investors, court, sec, charges, bahamas, extradition, a
    #4 track 21 S*=0.57  musk, twitter suspends, journalists
    #5 track 22 S*=0.14  talks, quantum, panama copper

  2022-12-20/2022-12-26
    #1 track 48 S*=0.72  boj, shock, shocks, 

## §7 — Validation (genAI lexicon post-hoc)

In [6]:
def pct_genai(headlines: list[str]) -> float:
    if not headlines:
        return 0.0
    return round(pd.Series(headlines).str.contains(GENAI_STANDARD, na=False).mean() * 100, 1)


val_rows = []
for c in track_records:
    hls = [news.at[i, "Headline"] for i in c["headline_indices"]]
    rep_hls = rep_df.loc[
        (rep_df.period == c["period"]) & (rep_df.track_id == c["track_id"]), "headline"
    ].tolist() if len(rep_df) else []
    rk = rank_df.loc[(rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"])].iloc[0]
    val_rows.append({
        "period": c["period"], "track_id": c["track_id"],
        "rank_in_week": int(rk["rank_in_week"]), "S_star": float(rk["S_star"]),
        "pct_standard_all": pct_genai(hls), "pct_standard_rep": pct_genai(rep_hls),
        "top_terms": ", ".join(c["top_terms"]), "first_seen": c["first_seen"],
        "headline_count": c["headline_count"],
    })
val_df = pd.DataFrame(val_rows).sort_values(["period", "rank_in_week"])
val_df.to_parquet(VALIDATION_PATH, index=False)

best = val_df.sort_values(["pct_standard_rep", "pct_standard_all", "S_star"], ascending=False).head(10)
print("Best genAI overlap (0.16):")
print(best[["period", "rank_in_week", "pct_standard_rep", "pct_standard_all", "S_star", "top_terms"]].to_string(index=False))

diag = []
for c in track_records:
    hits = [t for t in c["terms"] if GENAI_TERM_HINT.search(t)]
    if not hits:
        continue
    rk = rank_df.loc[(rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"])].iloc[0]
    diag.append({
        "period": c["period"], "track_id": c["track_id"], "rank_in_week": int(rk["rank_in_week"]),
        "S_star": float(rk["S_star"]), "genai_terms": ", ".join(sorted(hits)[:8]),
        "top_terms": ", ".join(c["top_terms"]),
    })
if diag:
    print("\nGenAI-vocabulary communities (diagnostic):")
    print(pd.DataFrame(diag).sort_values(["period", "rank_in_week"]).to_string(index=False))
else:
    print("\nNo GenAI-vocabulary communities found.")

top10_genai = [d for d in diag if d["rank_in_week"] <= 10]
print(f"\nGenAI-hint communities in top 10: {len(top10_genai)} / {len(diag)} diagnostic hits")

Best genAI overlap (0.16):
               period  rank_in_week  pct_standard_rep  pct_standard_all    S_star                                                                                                                         top_terms
2023-01-17/2023-01-23             1             100.0               6.5  1.102904                                            cuts, investment, job, microsoft, maker, openai, chatgpt, microsoft job, chatgpt maker
2022-12-20/2022-12-26            14               0.0               3.6 -0.259760                                                                                              sunday, nfl, ticket, google, youtube
2023-01-10/2023-01-16            16               0.0               1.9 -0.268025                                                                                                             takes, alibaba, cohen
2023-01-17/2023-01-23             7               0.0               1.3  0.280215                                            

## §8 — Method comparison vs 0.13 / 0.14

In [7]:
def best_rep_pct(validation_path: Path, week: str) -> float | None:
    if not validation_path.exists():
        return None
    v = pd.read_parquet(validation_path)
    sub = v.loc[v.period == week]
    if sub.empty:
        return None
    return float(sub["pct_standard_rep"].max())


def best_016_row(week: str) -> dict | None:
    sub = val_df.loc[val_df.period == week].sort_values(
        ["pct_standard_rep", "pct_standard_all", "S_star"], ascending=False
    )
    if sub.empty:
        return None
    r = sub.iloc[0]
    return {
        "best_016_rep_pct": r.pct_standard_rep,
        "best_016_rank": int(r.rank_in_week),
        "best_016_top_terms": r.top_terms[:120],
        "best_016_S_star": float(r.S_star),
    }


first_genai_rank = None
compare_rows = []
for week in discovery_weeks:
    row = {
        "week": week,
        "best_013_rep_pct": best_rep_pct(VALIDATION_013, week),
        "best_014_rep_pct": best_rep_pct(VALIDATION_014, week),
        "n_unseen_edges": int((edges_df.week == week).sum()),
        "n_communities": int((comm_df.period == week).sum()) if len(comm_df) else 0,
        "genai_lexicon_hits": int(news.loc[news.week == week, "is_genai"].sum()),
    }
    b16 = best_016_row(week)
    if b16:
        row.update(b16)
        if b16["best_016_rep_pct"] > 0 and first_genai_rank is None:
            first_genai_rank = {"week": week, "rank": b16["best_016_rank"]}
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows)
compare_df.to_parquet(COMPARE_PATH, index=False)
print(compare_df.to_string(index=False))
print(f"\nWrote {COMPARE_PATH.name}")
if first_genai_rank:
    print(f"First week with non-zero best_016_rep_pct: {first_genai_rank['week']} (rank {first_genai_rank['rank']})")

                 week  best_013_rep_pct  best_014_rep_pct  n_unseen_edges  n_communities  genai_lexicon_hits  best_016_rep_pct  best_016_rank                                                                                                       best_016_top_terms  best_016_S_star
2022-12-06/2022-12-12               0.0               0.0           22775             18                   4               0.0              1                                                                                                   novozymes, hansen, chr         0.680016
2022-12-13/2022-12-19               0.0               0.0           23606             17                   5               0.0              1                                            bank, ftx, fraud, danske, danske bank, guilty, forfeit, pleads guilty, pleads         1.285910
2022-12-20/2022-12-26               0.0               0.0           15630             17                   3               0.0             14                   

## §9 — Visualizations

In [12]:
# 1) Timeline — top 10 tracks by max S*
top_tracks = (
    rank_df.groupby("track_id")["S_star"].max().sort_values(ascending=False).head(10).index.tolist()
)
fig = go.Figure()
for tid in top_tracks:
    sub = rank_df.loc[rank_df.track_id == tid]
    fig.add_trace(go.Scatter(
        x=sub["period"].map(week_ts), y=sub["S_star"], mode="lines+markers",
        name=f"track {tid}", text=sub["top_terms"], hovertemplate="%{text}<extra></extra>",
    ))
add_date_marker(fig, CHATGPT_LAUNCH, "ChatGPT")
fig.update_layout(title="Top 10 S* tracks (unseen edge micro-graph)", height=480, xaxis_title="Week")
fig.write_html(TIMELINE_PATH)
print(f"Wrote {TIMELINE_PATH.name}")

# 2) Edge burst heatmap — top edges by max edge_score across discovery
edge_max = edges_df.groupby(["term_a", "term_b"])["edge_score"].max().sort_values(ascending=False)
top_pairs = edge_max.head(HEATMAP_TOP_EDGES).index.tolist()
pair_labels = [f"{a} — {b}" for a, b in top_pairs]
z = []
x_weeks = [week_ts(w) for w in discovery_weeks]
for a, b in top_pairs:
    row = []
    for week in discovery_weeks:
        sub = edges_df.loc[(edges_df.week == week) & (edges_df.term_a == a) & (edges_df.term_b == b), "edge_score"]
        row.append(float(sub.iloc[0]) if len(sub) else 0.0)
    z.append(row)
fig2 = go.Figure(data=go.Heatmap(x=x_weeks, y=pair_labels, z=z, colorscale="Viridis"))
fig2.update_layout(title=f"Top {HEATMAP_TOP_EDGES} unseen edges — edge_score by week", height=700)
fig2.write_html(HEATMAP_PATH)
print(f"Wrote {HEATMAP_PATH.name}")

# 3) Micro-graph for best genAI diagnostic week
best_diag_week = None
if diag:
    d0 = sorted(diag, key=lambda x: (x["rank_in_week"], -x["S_star"]))[0]
    best_diag_week = d0["period"]
if best_diag_week and best_diag_week in micro_graphs:
    g = micro_graphs[best_diag_week]
    pos = nx.spring_layout(g, seed=RANDOM_SEED, weight="weight")
    edge_x, edge_y = [], []
    for a, b in g.edges():
        x0, y0 = pos[a]
        x1, y1 = pos[b]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
    node_x = [pos[n][0] for n in g.nodes()]
    node_y = [pos[n][1] for n in g.nodes()]
    node_text = list(g.nodes())
    node_size = [8 + 4 * g.nodes[n].get("term_burst", 0) for n in g.nodes()]
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=0.5, color="#888"), hoverinfo="none"))
    fig3.add_trace(go.Scatter(
        x=node_x, y=node_y, mode="markers+text", text=node_text, textposition="top center",
        marker=dict(size=node_size, color="steelblue"), hovertext=node_text,
    ))
    fig3.update_layout(title=f"Micro-graph — {best_diag_week}", showlegend=False, height=600)
    fig3.write_html(MICROGRAPH_PATH)
    print(f"Wrote {MICROGRAPH_PATH.name} ({g.number_of_nodes()} nodes)")
else:
    print("Skipped micro-graph — no diagnostic week with graph")

# 4) Dashboard — unseen edges, communities, genAI hits
dash = compare_df.copy()
dash["week_start"] = dash["week"].map(week_ts)
fig4 = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=(
    "Unseen edges (count)", "Communities detected", "GenAI lexicon hits",
))
fig4.add_trace(go.Bar(x=dash["week_start"], y=dash["n_unseen_edges"], name="unseen edges"), row=1, col=1)
fig4.add_trace(go.Bar(x=dash["week_start"], y=dash["n_communities"], name="communities"), row=2, col=1)
fig4.add_trace(go.Bar(x=dash["week_start"], y=dash["genai_lexicon_hits"], name="genAI hits"), row=3, col=1)
add_date_marker(fig4, CHATGPT_LAUNCH, "ChatGPT")
fig4.update_layout(title="0.16 weekly dashboard (Dec 2022–Jan 2023)", height=700, showlegend=False)
fig4.write_html(DASHBOARD_PATH)
print(f"Wrote {DASHBOARD_PATH.name}")


from IPython.display import IFrame, display
for p in [TIMELINE_PATH, HEATMAP_PATH, MICROGRAPH_PATH, DASHBOARD_PATH]:
    display(IFrame(src=str(p), width="100%", height=600))

Wrote genai_unseen_timeline.html
Wrote genai_unseen_edge_heatmap.html
Wrote genai_unseen_micrograph.html (119 nodes)
Wrote genai_unseen_dashboard.html


## Interpretation

**Expected if pipeline works:** Jan 2023 weeks show a tracked micro-graph community with terms like `chatgpt`, `openai`, `microsoft`, `investment`, `bing`; edge heatmap brightens mid-Jan on `microsoft–openai` / `chatgpt–microsoft`.

**Dec 2022:** early/noisy signal only (single-digit `chatgpt` mentions per 0.15) — low rank is acceptable.

**Failure modes:** one-off macro pairs (Adani, FTX) may still rank high on individual weeks; check whether `WOW_MIN` and `MIN_INTERNAL_UNSEEN_EDGES` filter them. If top S* remains 0% genAI overlap, tune `TOP_EDGES_PER_WEEK` / `WOW_MIN` or add persistence gate (edge in ≥2 consecutive weeks).

**vs 0.13/0.14:** compare table in §8 — success = non-zero `best_016_rep_pct` and genAI-hint community in top 10 for at least one Jan week.